In [1]:
!pip -q install torch transformers datasets rouge-score sentencepiece accelerate

  Preparing metadata (setup.py) ... done


In [2]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW
from datasets import load_dataset
from rouge_score import rouge_scorer
from tqdm import tqdm
import numpy as np

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [4]:
CONFIG = {
    "model_name": "t5-base",
    "train_samples": 20000,
    "val_samples": 5000,

    "max_input_len": 512,
    "max_target_len": 128,

    "batch_size": 4,
    "gradient_accumulation": 4,

    "lr": 3e-5,
    "epochs": 3,
    "warmup_ratio": 0.1
}

In [5]:
tokenizer = T5Tokenizer.from_pretrained(CONFIG["model_name"])
model = T5ForConditionalGeneration.from_pretrained(CONFIG["model_name"])
model.gradient_checkpointing_enable()
model.to(device)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=768, out_features=3072, bias=False)
              (wo): Linear(in_features=3072, out_features=768, bias=False)
              (dropout): Dro

In [6]:
train_data = load_dataset(
    "cnn_dailymail",
    "3.0.0",
    split=f"train[:{CONFIG['train_samples']}]"
)

val_data = load_dataset(
    "cnn_dailymail",
    "3.0.0",
    split=f"validation[:{CONFIG['val_samples']}]"
)


README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

3.0.0/validation-00000-of-00001.parquet:   0%|          | 0.00/34.7M [00:00<?, ?B/s]

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

In [7]:
class T5SummaryDataset(Dataset):
    def __init__(self, data, tokenizer, config):
        self.data = data
        self.tokenizer = tokenizer
        self.config = config

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        article = self.data[idx]["article"]
        summary = self.data[idx]["highlights"]

        source_text = "summarize: " + article

        inputs = self.tokenizer(
            source_text,
            max_length=self.config["max_input_len"],
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )

        targets = self.tokenizer(
            summary,
            max_length=self.config["max_target_len"],
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )

        labels = targets["input_ids"].squeeze()
        labels[labels == tokenizer.pad_token_id] = -100

        return {
            "input_ids": inputs["input_ids"].squeeze(),
            "attention_mask": inputs["attention_mask"].squeeze(),
            "labels": labels
        }


In [8]:
train_dataset = T5SummaryDataset(train_data, tokenizer, CONFIG)
val_dataset = T5SummaryDataset(val_data, tokenizer, CONFIG)

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False
)

In [9]:
optimizer = AdamW(model.parameters(), lr=CONFIG["lr"])

total_steps = (
    len(train_loader)
    * CONFIG["epochs"]
    // CONFIG["gradient_accumulation"]
)

warmup_steps = int(total_steps * CONFIG["warmup_ratio"])

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    warmup_steps,
    total_steps
)


In [10]:
def train_one_epoch(model, loader):
    model.train()
    total_loss = 0
    optimizer.zero_grad()

    for step, batch in enumerate(tqdm(loader)):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss / CONFIG["gradient_accumulation"]
        loss.backward()

        if (step + 1) % CONFIG["gradient_accumulation"] == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item()

    return total_loss / len(loader)


In [11]:
def evaluate(model, loader, max_samples=100):
    model.eval()
    scorer = rouge_scorer.RougeScorer(
        ["rouge1", "rouge2", "rougeL"],
        use_stemmer=True
    )

    predictions, references = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            generated = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_length=128,
                num_beams=4
            )

            preds = tokenizer.batch_decode(
                generated,
                skip_special_tokens=True
            )

            labels[labels == -100] = tokenizer.pad_token_id
            refs = tokenizer.batch_decode(
                labels,
                skip_special_tokens=True
            )

            predictions.extend(preds)
            references.extend(refs)

            if len(predictions) >= max_samples:
                break

    r1, r2, rl = [], [], []
    for p, r in zip(predictions, references):
        scores = scorer.score(r, p)
        r1.append(scores["rouge1"].fmeasure)
        r2.append(scores["rouge2"].fmeasure)
        rl.append(scores["rougeL"].fmeasure)

    return np.mean(r1), np.mean(r2), np.mean(rl)


In [12]:
torch.cuda.empty_cache()

best_score = 0

for epoch in range(CONFIG["epochs"]):
    print(f"\nEpoch {epoch+1}")

    train_loss = train_one_epoch(model, train_loader)

    torch.cuda.empty_cache()

    rouge1, rouge2, rougeL = evaluate(model, val_loader)

    torch.cuda.empty_cache()

    avg_rouge = (rouge1 + rouge2 + rougeL) / 3

    print(f"Train Loss: {train_loss:.4f}")
    print(f"ROUGE-1: {rouge1:.4f}")
    print(f"ROUGE-2: {rouge2:.4f}")
    print(f"ROUGE-L: {rougeL:.4f}")

    if avg_rouge > best_score:
        best_score = avg_rouge
        model.save_pretrained("t5_summarizer/")
        tokenizer.save_pretrained("t5_summarizer/")



Epoch 1


100%|██████████| 5000/5000 [1:14:58<00:00,  1.11it/s]


Train Loss: 0.4042
ROUGE-1: 0.3604
ROUGE-2: 0.1582
ROUGE-L: 0.2688


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Epoch 2


100%|██████████| 5000/5000 [1:14:52<00:00,  1.11it/s]


Train Loss: 0.3802
ROUGE-1: 0.3563
ROUGE-2: 0.1581
ROUGE-L: 0.2620

Epoch 3


100%|██████████| 5000/5000 [1:14:48<00:00,  1.11it/s]


Train Loss: 0.3720
ROUGE-1: 0.3563
ROUGE-2: 0.1551
ROUGE-L: 0.2619


In [13]:
print("Saving model and tokenizer...")
model.save_pretrained("./t5_summarizer/")
tokenizer.save_pretrained("./t5_summarizer/")
print("savedd the model")

import os
saved_files = os.listdir("./t5_summarizer/")
print(f"Files in t5_summarizer: {saved_files}")


Saving model and tokenizer...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

savedd the model
Files in t5_summarizer: ['generation_config.json', 'tokenizer.json', 'model.safetensors', 'config.json', 'tokenizer_config.json']


### Model Usage

In [14]:
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration

In [15]:
MODEL_PATH = "t5_summarizer"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = T5Tokenizer.from_pretrained(MODEL_PATH)
model = T5ForConditionalGeneration.from_pretrained(MODEL_PATH)

model.to(device)
model.eval()


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=768, out_features=3072, bias=False)
              (wo): Linear(in_features=3072, out_features=768, bias=False)
              (dropout): Dro

In [16]:
def summarize(text, max_length=150, min_length=40):
    input_text = "summarize: " + text

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        summary_ids = model.generate(
            inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=max_length,
            min_length=min_length,
            num_beams=8,
            length_penalty=2.0,
            early_stopping=True,
            no_repeat_ngram_size=3,
            repetition_penalty=3.0,
            temperature=1.3,
            top_p=0.95,
            diversity_penalty=1.5,
            do_sample=False
        )

    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

In [17]:
article = """
Modern organizations operate within increasingly complex ecosystems shaped by rapid technological change, globalization, regulatory pressure, and shifting societal expectations. Decision-making is no longer confined to hierarchical command structures but is instead distributed across cross-functional teams that must integrate technical expertise, market intelligence, ethical considerations, and long-term strategic thinking. As digital systems generate vast quantities of data, leaders are challenged not only to collect information but to interpret it meaningfully, distinguishing signal from noise while avoiding cognitive biases that can distort judgment. The ability to synthesize diverse inputs into coherent strategies has therefore become a critical organizational capability, one that depends as much on communication and culture as on analytics and automation. Artificial intelligence has emerged as a transformative force across multiple industries, redefining how work is performed, value is created, and risk is managed. Machine learning systems now assist in tasks ranging from medical diagnosis and financial forecasting to supply chain optimization and natural language processing. However, the deployment of these systems introduces new complexities, including concerns about transparency, accountability, and fairness. Models trained on historical data may inadvertently reinforce existing inequalities, while highly optimized systems can behave in unexpected ways when exposed to novel conditions. As a result, organizations must balance the efficiency gains of automation with robust governance frameworks that ensure ethical alignment, regulatory compliance, and human oversight. Climate change represents another multidimensional challenge that intersects with economic development, public policy, and technological innovation. Rising global temperatures, extreme weather events, and resource scarcity are already disrupting infrastructure, agriculture, and population stability in many regions. In response, governments and corporations are investing in renewable energy, sustainable materials, and resilient system design. Yet progress is uneven, often constrained by political fragmentation, short-term economic incentives, and unequal access to capital. Addressing climate risk effectively requires coordinated action across national borders and industry sectors, as well as long-term planning horizons that extend beyond traditional electoral or quarterly business cycles. Education systems are also undergoing significant transformation as they adapt to digital delivery models, evolving labor market demands, and changing learner expectations. Online platforms and adaptive learning technologies have expanded access to knowledge, enabling personalized instruction at scale. At the same time, concerns persist regarding educational quality, student engagement, and the development of critical thinking skills in virtual environments. Employers increasingly emphasize competencies such as problem-solving, collaboration, and lifelong learning, placing pressure on institutions to move beyond rote instruction toward experiential and interdisciplinary approaches. The success of these reforms depends on aligning curriculum design, assessment methods, and faculty development with broader social and economic objectives. Finally, the interaction between technology, governance, and individual autonomy continues to provoke debate as societies grapple with issues of privacy, security, and digital identity. Surveillance capabilities have expanded alongside connectivity, raising questions about the appropriate balance between collective safety and personal freedom. Social media platforms influence public discourse at unprecedented scale, amplifying both constructive dialogue and misinformation. Policymakers face the difficult task of regulating rapidly evolving technologies without stifling innovation or entrenching incumbent power. Navigating this landscape requires nuanced understanding, evidence-based policymaking, and ongoing dialogue among technologists, regulators, and the public.
"""


In [18]:
summary = summarize(article)
print(summary)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Artificial intelligence has emerged as a transformative force across multiple industries . Climate change represents another multidimensional challenge that intersects with economic development, public policy, and technological innovation . Education systems are also undergoing significant transformation as they adapt to digital delivery models .


In [19]:
article="""
The evolution of complex systems in contemporary society reflects an ongoing tension between efficiency, adaptability, and control, as institutions increasingly rely on interconnected technological, economic, and social frameworks to function at scale. Advances in digital infrastructure have enabled real-time coordination across geographic boundaries, allowing organizations to optimize operations, monitor performance, and respond rapidly to changing conditions, yet this same interdependence has amplified systemic risk by reducing slack and increasing vulnerability to cascading failures. Financial markets, logistics networks, healthcare systems, and energy grids now depend on algorithmic decision-making and automated processes that can outperform human capabilities in speed and precision but often lack contextual awareness or moral reasoning. Consequently, small perturbations, whether triggered by software defects, policy misalignment, or unexpected external shocks, can propagate nonlinearly through these systems, producing outcomes that are difficult to predict or reverse. Addressing these challenges requires not only technical safeguards and redundancy but also institutional learning mechanisms that encourage transparency, cross-disciplinary collaboration, and continuous adaptation, ensuring that optimization does not come at the expense of resilience, trust, or long-term societal stability.
"""

In [20]:
summary = summarize(article)
print(summary)

Digital infrastructure has enabled real-time coordination across geographic boundaries . Interdependence has amplified systemic risk by reducing slack and increasing vulnerability to failures . Financial markets, logistics networks, healthcare systems depend on algorithmic decision-making .
